# 5.17 eval 就會停止計算梯度嗎？


Linear將[1,1]變成y=w1×1+w2×1+b，w1、w2是權重，b是偏置。切到eval後，還能對w1、w2、b求梯度嗎？能。評估模式控制前向行為，不負責關閉求導。

eval讓dropout這類訓練時隨機省略數值的操作改用評估行為；no_grad則讓此次計算不留下求導關係。先用沒有dropout的Linear看兩個開關。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch
from torch import nn

layer = nn.Linear(2, 1)
x = torch.ones(1, 2)
layer.eval()
output = layer(x)
print("eval輸出可求導", output.requires_grad, output.grad_fn)
output.sum().backward()
with torch.no_grad():
    other = layer(x)
print("no_grad輸出可求導", other.requires_grad, other.grad_fn)
assert output.requires_grad and not other.requires_grad

輸入一筆兩個1，Linear參數預設可求導。`eval`後output仍連着權重計算，所以打印True與一個 `grad_fn`；`grad_fn` 是記錄這個結果由哪種運算來的內部標記，不需背它的具體名稱。`backward()` 能把影響傳回權重。`no_grad`內的other則打印False與None，表示沒有這一條求導路線。

Linear本身沒有dropout，因此eval不會讓它數值變動；我們選它正是為了專注證明「評估行為」與「能否求導」獨立。換成含dropout的網絡時eval會影響輸出方式，仍不表示它禁用梯度。相反，訓練模式下使用no_grad也會不記錄那次運算。

第三個設定requires_grad=False只凍結相應參數。固定權重仍是可微配方，輸入若需要梯度，輸出仍能對它求導。固定參考常同時eval、凍結和no_grad，各有目的；若還需穿過它訓練前面的輸入產生器，就不能把該段包成no_grad。

練習先對 `layer.parameters()` 逐項執行 `parameter.requires_grad_(False)`，再把x建立時改成 `torch.ones(1,2, requires_grad=True)`。保留eval但不加no_grad，先預測輸出仍True、backward後x.grad有兩個數，分別等於固定的w1與w2：輸入各增加一點時，y分別按這兩個係數變化。再執行核對。你凍結的是哪一組旋鈕，不等於整段計算停止可微。

<details>
<summary>補充：實作約定與原始紀錄</summary>

短程式沿用本課工具與原計算語義。安裝、長訓練與重做操作見[訓練配方](https://birdhackor.github.io/tiny-perceptron-vlm/training.html)，不需要先完成長配方才能閱讀這個例子。

</details>
